# Production Agent — DynamoDB Cache (Local Testing)

Tests `production_agent.py` from `cache-dynamodb/02-production-agent/agent_files/` against the deployed DynamoDB table.

**Prerequisites:**
1. AWS credentials configured (`aws configure` or env vars)
2. `DynamoCacheStack` deployed (`cdk deploy` in `01-cache-layers-dynamodb/`)
3. Python packages: `uv pip install strands-agents boto3 matplotlib`

**The only external service is DynamoDB** — no Valkey, no VPC, no FT.* mock needed.

## Hook events wired by `ReasoningCacheHook`

| Event | Timing | What it does |
|-------|--------|-------------|
| `BeforeInvocationEvent` | Before first model call | KNN `search_vectors` for a similar past question; injects plan hint on hit |
| `BeforeToolCallEvent` | Before each tool | Exact-match `GetItem` lookup; swaps real tool with stub on hit |
| `AfterToolCallEvent` | After each tool | Stores fresh result with `PutItem`; records call in trajectory |
| `AfterInvocationEvent` | After agent loop | Stores question → trajectory with embedding for future plan hints |

**Tool result cache is model-agnostic** — key is `hash(tool_name + args)`, no model ID. A warm run with Claude reads tool results cached during a Nova cold run.

**DynamoDB TTL** is eventually consistent. For volatile tool results (e.g. `search_flights`, 5-min TTL), the cache code verifies the `ttl` attribute manually on every read.

In [ ]:
import sys, os, time
import boto3

sys.path.insert(0, "../02-production-agent/agent_files")
os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")

# Verify AWS credentials
sts = boto3.client("sts")
identity = sts.get_caller_identity()
print(f"Account: ...{identity['Account'][-4:]}  Region: {os.environ.get('AWS_DEFAULT_REGION')}")

# Read table config from SSM (written by DynamoCacheStack)
ssm = boto3.client("ssm")
TABLE_NAME   = ssm.get_parameter(Name="/dynamodb-cache/table-name")["Parameter"]["Value"]
VECTOR_INDEX = ssm.get_parameter(Name="/dynamodb-cache/vector-index-name")["Parameter"]["Value"]
GSI_NAME     = ssm.get_parameter(Name="/dynamodb-cache/entry-type-gsi-name")["Parameter"]["Value"]

os.environ["EMBEDDING_MODEL_ID"] = "amazon.titan-embed-text-v2:0"
os.environ["DUFFEL_SECRET_ARN"]  = ""

print(f"Table: {TABLE_NAME}  Index: {VECTOR_INDEX}")

# Single DynamoDB client — replaces the two Valkey clients
ddb = boto3.client("dynamodb")

## Agent setup

`ReasoningCacheHook` from `dynamodb_cache.py` connects to the deployed DynamoDB table.
No VPC, no FT.* workaround needed.

In [ ]:
from strands import Agent
from strands.models import BedrockModel
from tools import geocode_destination, climate_summary, wikipedia_summary
from dynamodb_cache import ReasoningCacheHook, flush_all

NOVA_MODEL_ID   = "us.amazon.nova-pro-v1:0"
CLAUDE_MODEL_ID = "us.anthropic.claude-3-5-haiku-20241022-v1:0"

SYSTEM_PROMPT = (
    "You are a travel research assistant. Use geocode_destination first to get "
    "coordinates, then climate_summary for weather data, and wikipedia_summary "
    "for visa policies. Maximum 4 sentences, plain text, in the user's language."
)

QUESTION = "What is the best time of year to visit Japan?"

### Run 1 — Nova Pro cold (empty cache)

Valkey is replaced by DynamoDB. All three tools execute real network calls.
Results are stored via `AfterToolCallEvent` → `PutItem` in DynamoDB.

⏳ Expect 15–45 s.

In [ ]:
cache_hook_nova_cold = ReasoningCacheHook(
    ddb_client=ddb,
    table_name=TABLE_NAME,
    vector_index=VECTOR_INDEX,
    entry_type_gsi=GSI_NAME,
    threshold=0.85,
    ttl=3600,
)
agent_nova = Agent(
    model=BedrockModel(model_id=NOVA_MODEL_ID),
    system_prompt=SYSTEM_PROMPT,
    tools=[geocode_destination, climate_summary, wikipedia_summary],
    hooks=[cache_hook_nova_cold],
)

start = time.time()
result_nova_cold = agent_nova(QUESTION)
nova_cold_ms = int((time.time() - start) * 1000)

usage = result_nova_cold.metrics.accumulated_usage
nova_cold_tokens = usage["totalTokens"]

print(f"⏱️  {nova_cold_ms} ms  —  📊 {nova_cold_tokens} tokens  —  🔄 {result_nova_cold.metrics.cycle_count} cycles")
print(f"   tool_cache_hits={cache_hook_nova_cold.stats.get('tool_cache_hits', 0)}  tool_executions={cache_hook_nova_cold.stats.get('tool_executions', 0)}")
print(f"\n{result_nova_cold}")

### Run 2 — Nova Pro warm (tool results cached in DynamoDB)

Same model, same question. `BeforeToolCallEvent` calls `GetItem` — tool results served from DynamoDB without hitting any external API.

In [ ]:
cache_hook_nova_warm = ReasoningCacheHook(
    ddb_client=ddb,
    table_name=TABLE_NAME,
    vector_index=VECTOR_INDEX,
    entry_type_gsi=GSI_NAME,
    threshold=0.85,
    ttl=3600,
)
agent_nova_warm = Agent(
    model=BedrockModel(model_id=NOVA_MODEL_ID),
    system_prompt=SYSTEM_PROMPT,
    tools=[geocode_destination, climate_summary, wikipedia_summary],
    hooks=[cache_hook_nova_warm],
)

start = time.time()
result_nova_warm = agent_nova_warm(QUESTION)
nova_warm_ms = int((time.time() - start) * 1000)

usage_warm = result_nova_warm.metrics.accumulated_usage
nova_warm_tokens = usage_warm["totalTokens"]

print(f"⏱️  {nova_warm_ms} ms  —  📊 {nova_warm_tokens} tokens  —  🔄 {result_nova_warm.metrics.cycle_count} cycles")
print(f"   tool_cache_hits={cache_hook_nova_warm.stats.get('tool_cache_hits', 0)}  tool_executions={cache_hook_nova_warm.stats.get('tool_executions', 0)}")

### Run 3 — Claude Haiku (reads Nova's cached tool results)

Different model, same question. The tool cache key is `hash(tool_name + args)` — no model ID in the key.
Claude gets the same `GetItem` hits from DynamoDB that Nova stored.

In [ ]:
cache_hook_claude = ReasoningCacheHook(
    ddb_client=ddb,
    table_name=TABLE_NAME,
    vector_index=VECTOR_INDEX,
    entry_type_gsi=GSI_NAME,
    threshold=0.85,
    ttl=3600,
)
agent_claude = Agent(
    model=BedrockModel(model_id=CLAUDE_MODEL_ID),
    system_prompt=SYSTEM_PROMPT,
    tools=[geocode_destination, climate_summary, wikipedia_summary],
    hooks=[cache_hook_claude],
)

start = time.time()
result_claude = agent_claude(QUESTION)
claude_ms = int((time.time() - start) * 1000)

usage_claude = result_claude.metrics.accumulated_usage
claude_tokens = usage_claude["totalTokens"]

print(f"⏱️  {claude_ms} ms  —  📊 {claude_tokens} tokens  —  🔄 {result_claude.metrics.cycle_count} cycles")
print(f"   tool_cache_hits={cache_hook_claude.stats.get('tool_cache_hits', 0)}  tool_executions={cache_hook_claude.stats.get('tool_executions', 0)}")
print(f"\n{result_claude}")

In [ ]:
import matplotlib, matplotlib.pyplot as plt
matplotlib.rcParams["figure.facecolor"] = "white"

labels  = ["Nova cold", "Nova warm", "Claude"]
tokens  = [nova_cold_tokens, nova_warm_tokens, claude_tokens]
latency = [nova_cold_ms,     nova_warm_ms,     claude_ms]
hits    = [
    cache_hook_nova_cold.stats.get("tool_cache_hits", 0),
    cache_hook_nova_warm.stats.get("tool_cache_hits", 0),
    cache_hook_claude.stats.get("tool_cache_hits", 0),
]
colors = ["#FF7043", "#42A5F5", "#7E57C2"]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, vals, title, unit in [
    (axes[0], tokens,  "Tokens",          ""),
    (axes[1], latency, "Latency",          " ms"),
    (axes[2], hits,    "Tool cache hits",  ""),
]:
    bars = ax.bar(labels, vals, color=colors, width=0.5)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + max(vals, default=1)*0.03,
                f"{v:,}{unit}", ha="center", fontsize=9)
    ax.set_title(title, fontweight="bold")
    ax.set_ylim(0, max(vals, default=1) * 1.35)
fig.suptitle("ReasoningCacheHook (DynamoDB) — Nova vs Claude, same cache", fontweight="bold")
plt.tight_layout()
plt.show()

print(f"{'Run':<22} {'Tokens':>8} {'Latency':>9} {'ToolHits':>9}")
print("-" * 52)
for label, tok, lat, hit in zip(labels, tokens, latency, hits):
    print(f"{label:<22} {tok:>8,} {lat:>7} ms {hit:>9}")

In [ ]:
# Flush DynamoDB table — removes all items
# Only use against the deployed DynamoCacheStack table
deleted = flush_all(ddb, TABLE_NAME)
print(f"Deleted {deleted} items from {TABLE_NAME}")